# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. We will:

- Load metadata and data records defined by the Croissant schema
- List available record sets and their `@id`s
- Load records from record sets into DataFrames
- Perform common exploratory data analysis (EDA) tasks
- Visualize the data

### Dataset Source
The dataset source is provided via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The URL of the Croissant schema for the FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset title:", getattr(metadata, 'name', None))
print("Description:", getattr(metadata, 'description', None))


## 2. Data Overview
Review available record sets, their fields, and associated `@id`s. This is crucial for referencing and extracting the correct data.

In [ ]:
# Find all available RecordSets and their structures in the metadata (referenced by @id)
record_sets = []
if hasattr(metadata, 'record_set'):
    record_sets = metadata.record_set
else:
    # Try typical alternate
    record_sets = getattr(metadata, 'recordSets', [])

if not record_sets:
    print("No record sets found in metadata (check schema or verify field). Trying to enumerate from dataset...")
    # Try to infer from available record set ids
    try:
        # The mlcroissant library's 'records()' method may yield available ids if given no input
        import itertools
        ids_found = set()
        # Attempt trial listing. Usually record sets are top-level fields, so we try some guesses.
        # Try common idioms and confirm with the dataset loader.
        for recset_name in [
                'cr:RecordSet:protein_records',  # Example
                'cr:RecordSet:main',
                'frontiers:protein_table',
                'cr:RecordSet',
                # You may need to adjust this if you know the actual record set@id
        ]:
            try:
                records = list(dataset.records(record_set=recset_name))
                if records:
                    print(f"Found record set: {recset_name}  (sample record: {records[0]})")
                    record_sets.append(recset_name)
            except Exception as e:
                continue
        if not record_sets:
            # Last resort: try standard table/data/data_table
            for recset_name in ['data', 'table', 'records', 'cr:recordSet', 'dataset_table']:
                try:
                    records = list(dataset.records(record_set=recset_name))
                    if records:
                        print(f"Found record set: {recset_name}  (sample record: {records[0]})")
                        record_sets.append(recset_name)
                except Exception:
                    continue
    except Exception as e:
        print("Could not enumerate record sets.", e)

# If record_sets is still empty, try to assume a default
if not record_sets:
    # Try a guess for FAIR^2 (from public reference, typical Croissant) - 'cr:RecordSet:ProteinTable'
    default_record_set = 'cr:RecordSet:ProteinTable'
    print(f"Using default record set ID: {default_record_set}")
    record_sets = [default_record_set]
else:
    print("Available record set @id(s):")
    for r in record_sets:
        print("-", r)

# Print out some records from the record set (using the first detected one)
print("\nSample records from a record set:")
sample_record_set = record_sets[0]
for i, record in enumerate(dataset.records(record_set=sample_record_set)):
    print(record)
    if i >= 2:
        break # Show only 3 records

## 3. Data Extraction
Load all records from the (main) record set into a DataFrame for further analysis.

We reference the record set, fields, and columns by their `@id`s.

In [ ]:
# Suppose we only found one main record set:
dataframes = {}

# You may select all or a subset of record sets depending on the dataset
for record_set_id in record_sets:
    # Load all records for this record set into a DataFrame
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set '{record_set_id}' with shape {df.shape}")
    else:
        print(f"No data found for record set {record_set_id}.")

# Show column names of the main DataFrame
main_record_set_id = record_sets[0]
if main_record_set_id in dataframes:
    print('Columns in DataFrame:', dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No DataFrame found for record set {main_record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
We now perform basic EDA on a numeric field.

- Select a numeric field by its column name (referenced by its Croissant `@id`)
- Filter records, normalize the field, and group by a categorical key if present.

> Replace `cr:Field:MW` with the correct field/column `@id` if your schema/overview uses a different one.

In [ ]:
# Choose one numeric field for demonstration — e.g., molecular weight (MW) or peptide count.
# Use column names as per DataFrame above; Croissant convention often is 'cr:Field:<name>' or a plain name.
# You may need to correct the field variable below if inspection above showed a different name.

main_df = dataframes[main_record_set_id]

# Try automatic selection of a likely numeric field
numeric_field_candidates = [c for c in main_df.columns if (c.endswith('MW') or c.lower().find('molecular_weight') >= 0 or c.lower().find('peptide') >= 0 or c.lower().find('count') >= 0)]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    # Fallback default
    numeric_field = main_df.columns[0]
    print(f"Falling back to field: {numeric_field}")

# Ensure numeric type
main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')

threshold = 10000  # Example threshold, change depending on data scale
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (e.g., description or accession)
group_field_candidates = [c for c in main_df.columns if ('description' in c.lower()) or ('accession' in c.lower()) or ('type' in c.lower())]
if group_field_candidates:
    group_field = group_field_candidates[0]
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (showing mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No suitable group-by field found.")

## 5. Visualization
Visualize the data distribution for the selected numeric field, and show how the normalization distributed the values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,5))
sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(12,5))
sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True, color='orange')
plt.title(f'Normalized {numeric_field} (Filtered, >{threshold})')
plt.xlabel(f'{numeric_field} (normalized)')
plt.ylabel('Frequency')
plt.show()


## 6. Conclusion
In this notebook, we illustrated how to use the `mlcroissant` library to load and analyze a research dataset defined by a Croissant schema. Key steps included:

- Inspecting available record sets and using their `@id` to load data
- Loading tabular data into pandas DataFrames
- Performing foundational EDA, such as filtering and normalizing numeric data, and visualizing distributions

This approach can be extended for more complex tasks and applied to other FAIR-compliant datasets described with Croissant schemas.

*Remember: always reference dataset entities using their `@id`, as done in this workflow for full reproducibility and interpretability!*